# Structure-Revealing vs Structure-Using Methods

Structural tracking distinguishes between two categories of DataFrame/Series methods:

## Structure-Revealing Methods (TRACKED)
Methods whose **primary purpose** is to reveal structure:
- Attribute access: `.columns`, `.shape`, `.index`, `.dtypes`
- Introspection: `.describe()`, `.info()`, `.head()`, `.tail()`
- Special: `len(df)`, `for col in df:`

## Structure-Using Methods (NOT TRACKED)
Methods that **internally access** structure as implementation detail:
- Mutation: `df['x'] = value`, `del df['x']`, `.insert()`, `.drop()`
- Access: `df['x']`, `df.loc[]`, `df.iloc[]`
- Arithmetic: `df + 1`, `df * 2`, `df - other`
- Aggregation: `.mean()`, `.sum()`, `.min()`, `.max()`
- Transform: `.apply()`, `.groupby()`, `.fillna()`
- Display: `repr(df)`, `str(df)`, `df._repr_html_()`

In [ ]:
import pandas as pd
import numpy as np

# Enable enforce mode
%structural_tracking enforce

## Part 1: Structure-Revealing Operations

These operations are **tracked** because their primary purpose is to reveal structure.

In [ ]:
# Create a DataFrame
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})

In [ ]:
# TRACKED: Explicit structural access
print(f"Columns: {list(df.columns)}")
print(f"Shape: {df.shape}")
print(f"Number of rows: {len(df)}")

## Part 2: Structure-Using Operations

These operations are **NOT tracked** even though they internally access `.columns`, `.index`, etc.

In [ ]:
# Create a fresh DataFrame
df2 = pd.DataFrame({'x': [1, 2, 3], 'y': [4, 5, 6]})

In [ ]:
# NOT TRACKED: Setting a column (internally accesses .columns for broadcasting)
df2['z'] = [7, 8, 9]
print("Column added - no structural read recorded!")

In [ ]:
# NOT TRACKED: Getting a column
values = df2['x']
print(f"Got values: {list(values)}")

In [ ]:
# NOT TRACKED: Arithmetic (internally accesses structure for alignment)
result = df2 + 10
print(f"Arithmetic result shape: {result.shape}")

In [ ]:
# NOT TRACKED: Aggregation (internally accesses structure)
means = df2.mean()
sums = df2.sum()
print(f"Means: {list(means)}")
print(f"Sums: {list(sums)}")

In [ ]:
# NOT TRACKED: Transform operations
doubled = df2.apply(lambda x: x * 2)
sorted_df = df2.sort_values('x')
print("Transform operations - no structural reads!")

In [ ]:
# NOT TRACKED: Display (internally accesses .columns, .index for rendering)
_ = repr(df2)  # Used for printing
_ = str(df2)   # String conversion
print("Display operations - no structural reads!")

## Part 3: Why This Matters for SDC

**Key insight**: We only want to track **explicit user access** to structure.

When you write `print(df.columns)`, you explicitly asked about the columns.

When you write `df['x'] = 3`, you're modifying data - you didn't ask about structure,
even though pandas internally checks structure during the operation.

In [ ]:
# Create a fresh DataFrame
data = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})

In [ ]:
# This cell modifies data
# It does NOT record structural reads (even though __setitem__ accesses structure internally)
data['c'] = [7, 8, 9]

In [ ]:
# This cell also modifies data - still no structural reads
data['d'] = data['a'] + data['b']

In [ ]:
# NOW we explicitly read structure - this IS tracked
print(f"Final columns: {list(data.columns)}")
print(f"Final shape: {data.shape}")

## Part 4: Comparison Table

| Operation | Category | Tracked? | Example |
|-----------|----------|----------|---------|
| `df.columns` | Structure-revealing | Yes | Explicit attribute access |
| `df.shape` | Structure-revealing | Yes | Explicit attribute access |
| `len(df)` | Structure-revealing | Yes | Explicit row count |
| `df.describe()` | Structure-revealing | Yes | Introspection method |
| `df['x'] = 3` | Structure-using | No | Mutation |
| `df['x']` | Structure-using | No | Column access |
| `df + 1` | Structure-using | No | Arithmetic |
| `df.mean()` | Structure-using | No | Aggregation |
| `repr(df)` | Structure-using | No | Display |
| `df.apply(fn)` | Structure-using | No | Transform |

In [ ]:
# Reset to warn mode
%structural_tracking warn
print("Demo complete!")